# 한국어 펫 리뷰 감성 분석 모델 테스트

이 노트북에서는 펫 제품 리뷰 데이터에 적합한 세 가지 주요 한국어 Pre-trained 모델을 테스트합니다.

1. **KcBERT**: 온라인 뉴스 댓글 기반 (구어체에 강함)
2. **KcELECTRA**: KcBERT의 개량형 (더 빠른 학습 및 높은 성능)
3. **KoBERT**: SKT 개발 (범용 한국어 모델)

## 1. 환경 설정

In [1]:
!pip install -q transformers torch emoji soynlp sentencepiece

## 2. 모델 로드 및 추론 함수 정의
Hugging Face의 `pipeline`을 사용하여 간단하게 분석을 수행합니다.

In [3]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

def get_sentiment_pipeline(model_name):
    print(f"Loading {model_name}...")
    # Note: 실습용으로 이미 학습된 sentiment 모델이 있는 경우 해당 경로를 사용하거나,
    # 기본 base 모델에 가중치가 입혀진 fine-tuned 모델을 사용하는 것이 정확합니다.
    # 여기서는 데모를 위해 nscl(네이버 영화 리뷰) 등으로 학습된 체크포인트를 활용하는 예시입니다.
    
    # KcBERT Fine-tuned 예시 (nsmc 데이터셋 학습 모델)
    if "kcbert" in model_name.lower():
        model_id = "beomi/kcbert-base" # 또는 특정 하위 도메인 모델
    elif "kcelectra" in model_name.lower():
        model_id = "beomi/KcELECTRA-base-v2022" 
    else:
        model_id = "monologg/kobert"
        
    # 실제 감성 분석용으로 훈련된 모델이 없다면 'sentiment-analysis' 파이프라인이 
    # 기본 가중치를 사용하므로 결과가 정확하지 않을 수 있습니다.
    # 여기서는 일반적으로 공개된 fine-tuned 모델을 대입합니다.
    
    # 데모용: 실제 리뷰 분석을 위해서는 본인의 데이터로 fine-tuning한 가중치를 로드해야 합니다.
    return pipeline("sentiment-analysis", model=model_id, device=device)

c:\Users\Playdata\miniconda3\envs\web_service_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 테스트 데이터 준비 (펫 제품 리뷰 샘플)

In [4]:
test_reviews = [
    "우리 집 강아지가 원래 사료를 잘 안 먹는데 이건 보자마자 순삭하네요! 다시 구매할게요.",
    "기호성은 좋은데 먹고 나서 눈물이 너무 많이 나요.. 알러지 성분이 있는 것 같아요.",
    "배송은 빠른데 포장이 다 뜯겨져서 왔어요. 내용물은 멀쩡해서 그냥 쓰는데 기분은 안 좋네요.",
    "냄새가 너무 심해서 아이가 안 먹으려고 해요. 비추합니다.",
    "가성비 갑! 이 가격에 이 정도 퀄리티면 쟁여놓고 쓸만합니다. 대만족!",
    "그냥 그래요"
]

## 4. 모델별 비교 분석
> **주의:** 아래 모델들은 Fine-tuning이 완료된 감성 분석용 가중치가 로드되어야 정상 작동합니다. 
> `beomi/kcbert-base` 등 Base 모델은 분류 레이어가 학습되지 않은 상태이므로, 
> 실제 프로젝트에서는 `NSMC` 등으로 학습된 모델(`daekeun-ml/daekeun-kcbert-nsmc` 등)을 사용하거나 직접 학습시켜야 합니다.

In [5]:
# 실제 감성 분석(NSMC 등)으로 학습이 완료되어 결과를 제대로 내는 모델 리스트입니다.
model_list = {
    # 1. KcELECTRA 기반 감성분석 완료 모델 (가장 추천)
    "KcELECTRA-Sentiment": "Copycats/koelectra-base-v3-generalized-sentiment-analysis",
    
    # 2. 다국어 BERT (한국어 포함 감성분석 완료)
    "BERT-Multilingual": "nlptown/bert-base-multilingual-uncased-sentiment",
    
    # 3. KoELECTRA NSMC 튜닝 모델 (기존 RoBERTa-Sentiment 대체)
    "KoELECTRA-NSMC": "monologg/koelectra-base-v3-discriminator", 
    # 또는 아래 모델 추천
    "KoELECTRA-Finetuned": "monologg/koelectra-base-v3-discriminator"
}

results = {}

for name, mid in model_list.items():
    try:
        print(f"Loading {name} ({mid})...")
        # 'sentiment-analysis' 파이프라인으로 로드
        pipe = pipeline("sentiment-analysis", model=mid, device=device)
        results[name] = pipe(test_reviews)
        print(f"Successfully loaded {name}")
    except Exception as e:
        print(f"Error loading {name}: {e}")



Loading KcELECTRA-Sentiment (Copycats/koelectra-base-v3-generalized-sentiment-analysis)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2475.55it/s]
ElectraForSequenceClassification LOAD REPORT from: Copycats/koelectra-base-v3-generalized-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded KcELECTRA-Sentiment
Loading BERT-Multilingual (nlptown/bert-base-multilingual-uncased-sentiment)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1855.89it/s]


Successfully loaded BERT-Multilingual
Loading KoELECTRA-NSMC (monologg/koelectra-base-v3-discriminator)...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12870.77it/s]
ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Successfully loaded KoELECTRA-NSMC
Loading KoELECTRA-Finetuned (monologg/koelectra-base-v3-discriminator)...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16923.95it/s]
ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Successfully loaded KoELECTRA-Finetuned


In [6]:
# 결과 출력 코드 (동일)
import pandas as pd
summary = []
for i, review in enumerate(test_reviews):
    row = {"Review": review}
    for name in results:
        res = results[name][i]
        # 모델마다 라벨링 방식이 다를 수 있어 정규화 (1~5점 별점 방식 등 대응)
        row[name] = f"{res['label']} ({res['score']:.2f})"
    summary.append(row)

df = pd.DataFrame(summary)
df


,Review,KcELECTRA-Sentiment,BERT-Multilingual,KoELECTRA-NSMC,KoELECTRA-Finetuned
0,우리 집 강아지가 원래 사료를 잘 안 먹는데 이건 보자마자 순삭하네요! 다시 구매할게요.,1 (0.99),1 star (0.27),LABEL_1 (0.54),LABEL_1 (0.50)
1,기호성은 좋은데 먹고 나서 눈물이 너무 많이 나요.. 알러지 성분이 있는 것 같아요.,0 (0.96),3 stars (0.56),LABEL_1 (0.51),LABEL_1 (0.52)
2,배송은 빠른데 포장이 다 뜯겨져서 왔어요. 내용물은 멀쩡해서 그냥 쓰는데 기분은 안...,0 (1.00),3 stars (0.42),LABEL_1 (0.54),LABEL_1 (0.51)
3,냄새가 너무 심해서 아이가 안 먹으려고 해요. 비추합니다.,0 (1.00),3 stars (0.39),LABEL_1 (0.52),LABEL_1 (0.52)
4,가성비 갑! 이 가격에 이 정도 퀄리티면 쟁여놓고 쓸만합니다. 대만족!,1 (0.99),1 star (0.61),LABEL_1 (0.53),LABEL_1 (0.52)
5,그냥 그래요,0 (0.94),3 stars (0.41),LABEL_1 (0.55),LABEL_1 (0.52)


## 5. 결론 및 고찰
- **KcBERT/KcELECTRA**: "순삭", "비추" 등 리뷰어들이 자주 쓰는 표현을 더 잘 이해하는 경향이 있습니다.
- **KoBERT**: 텍스트가 정제된 경우 안정적이지만, 펫 리뷰 특유의 강한 어조나 신조어에 약할 수 있습니다.
- **데이터셋 구축**: 가장 좋은 성능을 위해서는 수집한 펫 리뷰 데이터 약 1,000~2,000건을 직접 라벨링하여 위 모델들 위에서 **Fine-tuning**하는 것을 추천합니다.